In [3]:
import pandas as pd
import numpy as np

# Configuration
INPUT_FILE = "BigSolDB 2.0 Dataset.csv"

def run_eda():
    print(f"Loading {INPUT_FILE} for EDA...")
    
    # Load specific columns to check quality
    try:
        df = pd.read_csv(INPUT_FILE, usecols=['Solvent', 'SMILES_Solvent', 'LogS(mol/L)'])
    except FileNotFoundError:
        print(f"Error: File not found at {INPUT_FILE}")
        return

    total_rows = len(df)
    print(f"Total rows: {total_rows}")

    # 1. Check for missing Solvent SMILES
    # We check for standard NaNs AND the "-" string you mentioned
    missing_mask = df['SMILES_Solvent'].isna() | (df['SMILES_Solvent'].astype(str).str.strip() == '-')
    missing_smiles = df[missing_mask]
    unique_missing_solvents = missing_smiles['Solvent'].unique()

    print(f"\n--- Solvent SMILES Analysis ---")
    print(f"Rows with missing/hyphenated Solvent SMILES: {len(missing_smiles)}")
    
    if len(unique_missing_solvents) > 0:
        print(f"Unique Solvents affecting {len(missing_smiles)} rows:")
        print(unique_missing_solvents)
    else:
        print("Clean! All solvents have valid SMILES strings.")

    # 2. Check LogS availability
    # We check for NaNs in the provided LogS column
    missing_logs = df[df['LogS(mol/L)'].isna()]
    
    print(f"\n--- LogS Target Analysis ---")
    print(f"Rows with pre-calculated LogS: {total_rows - len(missing_logs)}")
    print(f"Rows MISSING LogS (need calculation): {len(missing_logs)}")

if __name__ == "__main__":
    run_eda()

Loading BigSolDB 2.0 Dataset.csv for EDA...
Total rows: 103944

--- Solvent SMILES Analysis ---
Rows with missing/hyphenated Solvent SMILES: 299
Unique Solvents affecting 299 rows:
['PEG-400' 'PEG-200' 'PEGDME 250' 'PEG-600' 'PEG-300']

--- LogS Target Analysis ---
Rows with pre-calculated LogS: 100983
Rows MISSING LogS (need calculation): 2961


In [6]:
import pandas as pd
from thermo.chemical import Chemical
import warnings

# Configuration
INPUT_FILE = "./BigSolDB 2.0 Dataset.csv"

# Current Alias Map (Add to this based on the output below)
SOLVENT_ALIASES = {
    "THF": "tetrahydrofuran",
    "n-heptane": "heptane",
    "DMS": "methylthiomethane",
    "2-ethyl-n-hexanol": "2-Ethyl hexanol",
    "3,6-dioxa-1-decanol": "butoxyethoxyethanol",
    "DEF": "diethylformamide",
    "ethanol": "ethanol",
    "methanol": "methanol",
    "water": "water"
}

def check_thermo_compatibility():
    print(f"Loading {INPUT_FILE}...")
    # We need Solubility(mole_fraction) to see if calculation is even possible
    use_cols = ['Solvent', 'LogS(mol/L)', 'Solubility(mole_fraction)', 'Temperature_K']
    df = pd.read_csv(INPUT_FILE, usecols=use_cols)
    
    # 1. Isolate rows that NEED calculation
    # (LogS is Missing AND Mole Fraction is available)
    calc_needed = df[df['LogS(mol/L)'].isna() & (df['Solubility(mole_fraction)'] > 0)]
    
    unique_solvents = calc_needed['Solvent'].unique()
    print(f"\nTotal rows needing calculation: {len(calc_needed)}")
    print(f"Unique solvents involved: {len(unique_solvents)}")
    
    if len(unique_solvents) == 0:
        print("Good news! No backfilling needed. All valid mole fractions already have LogS.")
        return

    print("\n--- Testing Thermo Library Compatibility ---")
    print(f"Checking {len(unique_solvents)} solvents against `thermo` database...")
    
    failed_solvents = []
    success_solvents = []
    
    # Suppress thermo warnings for clean output
    warnings.filterwarnings("ignore")
    
    for name in unique_solvents:
        # Use alias if strictly defined, otherwise use raw name
        lookup_name = SOLVENT_ALIASES.get(name, name)
        
        try:
            # We assume standard temp (298K) just for recognition check
            chem = Chemical(lookup_name, T=298.15)
            
            # Check if critical properties exist
            if chem.MW is None or chem.rho is None:
                failed_solvents.append(name)
            else:
                success_solvents.append(name)
        except Exception:
            failed_solvents.append(name)

    # Output Results
    print(f"\nPassed: {len(success_solvents)}")
    print(f"Failed: {len(failed_solvents)}")
    
    if len(failed_solvents) > 0:
        print("\n[ACTION REQUIRED] The following solvents failed lookup.")
        print("Add them to your SOLVENT_ALIASES map if you want to save these rows:")
        print("-" * 60)
        for s in failed_solvents:
            print(f'"{s}": "???",')
        print("-" * 60)
    else:
        print("\nAll solvents recognized! No alias updates needed.")

if __name__ == "__main__":
    check_thermo_compatibility()

Loading ./BigSolDB 2.0 Dataset.csv...

Total rows needing calculation: 2961
Unique solvents involved: 143

--- Testing Thermo Library Compatibility ---
Checking 143 solvents against `thermo` database...

Passed: 134
Failed: 9

[ACTION REQUIRED] The following solvents failed lookup.
Add them to your SOLVENT_ALIASES map if you want to save these rows:
------------------------------------------------------------
"PEG-200": "???",
"ε-caprolactone": "???",
"PEGDME 250": "???",
"2-methyl-cyclohexyl acetate": "???",
"diisobutyl methanol": "???",
"propanediol butyl ether": "???",
"span 80": "???",
"PEG-600": "???",
"PEG-300": "???",
------------------------------------------------------------


In [8]:
import pandas as pd
from thermo.chemical import Chemical
import warnings

# Configuration
INPUT_FILE = "./BigSolDB 2.0 Dataset.csv"

# The specific solvents you want to fix
TARGET_SOLVENTS = [
    "ε-caprolactone",
    "2-methyl-cyclohexyl acetate",
    "diisobutyl methanol",
    "propanediol butyl ether",
    "span 80"
]

def find_aliases():
    print(f"Scanning {INPUT_FILE} for target solvents...")
    
    # Load only necessary columns
    try:
        df = pd.read_csv(INPUT_FILE, usecols=['Solvent', 'SMILES_Solvent'])
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    # Filter for our targets
    subset = df[df['Solvent'].isin(TARGET_SOLVENTS)].drop_duplicates(subset=['Solvent'])
    
    if len(subset) == 0:
        print("Error: None of the target solvents were found in the CSV!")
        return

    print(f"\nFound {len(subset)} targets. Checking thermo database...\n")
    warnings.filterwarnings("ignore")
    
    found_aliases = {}
    failed_solvents = []

    for _, row in subset.iterrows():
        original_name = row['Solvent']
        smiles = row['SMILES_Solvent']
        
        # 1. Try to find the official name using SMILES
        try:
            # Check if SMILES is valid
            if pd.isna(smiles) or str(smiles).strip() in ["-", ""]:
                raise ValueError("Missing SMILES")

            c = Chemical(smiles, T=298.15)
            
            # Verify we got valid physics back
            if c.MW is not None and c.rho is not None:
                found_aliases[original_name] = c.name
                print(f"✅ MATCH: '{original_name}' -> '{c.name}'")
            else:
                failed_solvents.append((original_name, "Thermo found object but missing MW/Rho"))
                
        except Exception as e:
            # Special case for Span 80 (often a mixture)
            failed_solvents.append((original_name, f"Thermo lookup failed ({e})"))

    # --- OUTPUT ---
    print("\n" + "="*40)
    print("COPY THIS INTO clean_bigsol.py")
    print("="*40)
    print("SOLVENT_ALIASES = {")
    
    # Print the ones you already had (for context)
    print('    "THF": "tetrahydrofuran",')
    print('    "n-heptane": "heptane",')
    # ... etc ...
    
    # Print the NEW findings
    for old, new in found_aliases.items():
        print(f'    "{old}": "{new}",')
        
    print("}")
    
    if len(failed_solvents) > 0:
        print("\n" + "!"*40)
        print("WARNING: These solvents failed lookup.")
        print("You should probably DROP them (likely mixtures/polymers).")
        print("!"*40)
        for name, reason in failed_solvents:
            print(f"- {name}: {reason}")

if __name__ == "__main__":
    find_aliases()

Scanning ./BigSolDB 2.0 Dataset.csv for target solvents...

Found 5 targets. Checking thermo database...


COPY THIS INTO clean_bigsol.py
SOLVENT_ALIASES = {
    "THF": "tetrahydrofuran",
    "n-heptane": "heptane",
}

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
You should probably DROP them (likely mixtures/polymers).
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
- ε-caprolactone: Thermo lookup failed (Chemical name (O=C1CCCCCO1) not recognized)
- 2-methyl-cyclohexyl acetate: Thermo lookup failed (Chemical name (CC(=O)OC1CCCCC1C) not recognized)
- diisobutyl methanol: Thermo lookup failed (Chemical name (C(C(C)C)C(O)CC(C)C) not recognized)
- propanediol butyl ether: Thermo lookup failed (Chemical name (CCCCOCC(O)CO) not recognized)
- span 80: Thermo lookup failed (Chemical name (CCCCCCCC=CCCCCCCCC(=O)O[C@@H]1[C@H](O)[C@H](CO)OC[C@H]1O) not recognized)


In [1]:
!python clean_bigsol.py --input "BigSolDB 2.0 Dataset.csv"

Cleaning BigSolDB...
Canonicalizing Solutes (Tautomer Standard)...
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12947 |     MMMMMMM   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                          |        0 /    12948 |      
   0.00%                                 

In [2]:
import pandas as pd

# Load the newly cleaned BigSolDB 2.0
df = pd.read_csv("./bigsol_clean.csv")

# 1. Get counts per solvent
counts = df['Solvent'].value_counts()
total_rows = len(df)

# 2. Calculate cumulative coverage
stats = pd.DataFrame({
    'Solvent': counts.index,
    'Count': counts.values
})
stats['Cumulative_Count'] = stats['Count'].cumsum()
stats['Percentage_of_Data'] = (stats['Cumulative_Count'] / total_rows) * 100

# 3. Find the N that gets closest to 80%
n_80 = (stats['Percentage_of_Data'] - 80).abs().idxmin() + 1
coverage = stats.iloc[n_80-1]['Percentage_of_Data']

print(f"--- Analysis for BigSolDB 2.0 (N={len(counts)} unique solvents) ---")
print(f"Targeting ~80% inclusion for training...")
print(f"Recommended TOP_N_SOLVENTS: {n_80}")
print(f"This includes {stats.iloc[n_80-1]['Cumulative_Count']} rows ({coverage:.2f}%)")
print(f"The remaining {total_rows - stats.iloc[n_80-1]['Cumulative_Count']} rows ({100-coverage:.2f}%) will be in the OOD Test set.")

# Optional: Show the top 10 rows of the stats table to see the distribution
# display(stats.head(10))

--- Analysis for BigSolDB 2.0 (N=198 unique solvents) ---
Targeting ~80% inclusion for training...
Recommended TOP_N_SOLVENTS: 21
This includes 71648 rows (80.10%)
The remaining 17803 rows (19.90%) will be in the OOD Test set.
